In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import string
from itertools import product
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import matplotlib.pyplot as plt

In [ ]:
# Baixar os recursos necessários do NLTK
nltk.download('punkt')
nltk.download('stopwords')


In [ ]:
# Carregar os dados originais
df_nomes = pd.read_csv('nomes_pessoais.csv')
df_nomes['y'] = 1

df_palavras = pd.read_csv('palavras_comuns.csv').sample(n=102561, random_state=42)
df_palavras['y'] = 0

df = pd.concat([df_nomes, df_palavras])
df.shape

In [ ]:
# Carregar o arquivo gabarito_formatado.csv
unique_words_df = pd.read_csv('gabarito_formatado.csv')

# Remover valores NaN na coluna 'palavra'
unique_words_df = unique_words_df.dropna(subset=['palavra'])

# Converter todos os valores '-1' para '0' na coluna 'y'
unique_words_df['y'] = unique_words_df['y'].replace(-1, 0)

words_with_negative_label = unique_words_df[unique_words_df['y'] == -1]

words_with_negative_label_list = words_with_negative_label['palavra'].tolist()

print(words_with_negative_label_list)

In [ ]:
# Usar o alfabeto e bigrams
alfabeto = list(string.ascii_uppercase)
alfabeto_bigrams = [''.join(bigram) for bigram in list(product(alfabeto, repeat=2))]

vectorizer = CountVectorizer(analyzer='char', vocabulary=alfabeto_bigrams, lowercase=False, ngram_range=(2, 2))

In [ ]:
# Vetorizar os dados de treino completos
X_train_vector = vectorizer.fit_transform(df['X'])
y_train = df['y']

# Vetorizar as palavras únicas para teste
X_unique_words_vector = vectorizer.transform(unique_words_df['palavra'])
y_true = unique_words_df['y']


In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train_vector, y_train)

In [ ]:
y_pred = rf.predict(X_unique_words_vector)

# Calcular as métricas
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

In [ ]:
# Calcular a matriz de confusão
cm = confusion_matrix(y_true, y_pred)
fn = cm[1, 0]  # Falsos Negativos


In [ ]:
# Dicionário para armazenar os falsos negativos
false_negatives = {}

# Encontrar os falsos negativos
fn_indices = [i for i, (true, pred) in enumerate(zip(y_true, y_pred)) if true == 1 and pred == 0]
false_negatives = unique_words_df.iloc[fn_indices]['palavra'].tolist()